# 06— CNN-BiLSTM Downstream Evaluation, Project-Adapted


Kept from the original script:

```text
CNN-BiLSTM architecture
native-rate branch logic
training loop
validation logic
experiment design
metrics
saved-result style
real_to_synthetic uses saved Real->Real models when USE_REAL_MODELS=True
```

Changed only for your project:

```text
REAL_DIR -> data/processed/native_rates/
SYN_DIR  -> data/synthetic_subjects/<model_family>/<method>/
OUT_DIR  -> results/downstream_cnnbilstm/<model_family>/<method>/
PRETRAINED_RESULTS_DIR -> models/downstream/pretrained/cnnbilstm_native_results/
```

The first code cell only defines the code. The final cell runs it.


In [3]:
# ============================================================
# evaluate_cnnbilstm_native_like_tsai_aeon.py
#
# CNN-BiLSTM downstream HAR evaluation with the SAME experiment
# layout, saved-result style, per-activity reports, and reusable
# Real->Real model behavior as the tsai/aeon script.
#
# Real input:
#   processed_all_subjects_native_rates/
#       all_X_acc_32hz.npy        [N, 256, 3]
#       all_X_bvp_64hz.npy        [N, 512, 1]
#       all_X_slow_4hz.npy        [N, 32, 2]   # [:,:,0]=EDA, [:,:,1]=TEMP
#       all_y.npy
#       all_subject.npy
#
# Synthetic input:
#   CNN_clean_v1/
#       generated_subjects_X_acc_32hz.npy
#       generated_subjects_X_bvp_64hz.npy
#       generated_subjects_X_slow_4hz.npy
#       generated_subjects_all_y.npy
#       generated_subjects_all_subject.npy
#
# Experiments per modality:
#   1. real_to_real
#   2. real_to_synthetic        # uses saved Real->Real model if USE_REAL_MODELS=True
#   3. synthetic_to_real
#   4. real_plus_synthetic_to_real
#
# Modalities:
#   acc, bvp, eda, temp, fused
#
# Important:
#   - acc/bvp/eda/temp are true individual native-rate models.
#   - fused is native multirate fusion using separate ACC, BVP, EDA, TEMP branches.
#   - It does NOT upsample EDA/TEMP for the CNN-BiLSTM fused model.
# ============================================================

from pathlib import Path
import copy
import itertools
import json
import random
import re
import warnings
from contextlib import nullcontext

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)


# =========================
# MAIN RUN KNOBS
# =========================

# If True, only trains/saves Real->Real models and metrics.
# Use this first if you want to share the saved real models with others.
RUN_REAL_TO_REAL_ONLY = False

# If True, real_to_synthetic loads the saved Real->Real model instead of retraining.
# Same behavior as your tsai/aeon script.
USE_REAL_MODELS = True

# Same default modality grid as the tsai/aeon script.
# Individual EDA and TEMP are separate models.
MODALITIES = ["acc", "bvp", "eda", "temp", "fused"]


# =========================
# BASIC SETTINGS
# =========================

CNN_MODEL_NAME = "NativeCNNBiLSTM"

SEED = 42
SYN_TEST_N_SUBJECTS = 10

# =========================
# PROJECT PATH ADAPTATION
# =========================
# Only the folder paths are adapted here.
# The CNN-BiLSTM model, training loop, experiments, metrics, and saving logic are kept from the original script.

PROJECT_ROOT = Path("/home/iailab42/khans1/projects/ir")

# Choose one synthetic source, then run the notebook/script.
# Options:
#   KoVAE:   SELECTED_MODEL_FAMILY = "kovae",   SELECTED_SYNTHETIC_METHOD = "rollout_v1"
#   KoVAE:   SELECTED_MODEL_FAMILY = "kovae",   SELECTED_SYNTHETIC_METHOD = "posterior_bank_v2"
#   TimeVAE: SELECTED_MODEL_FAMILY = "timevae", SELECTED_SYNTHETIC_METHOD = "prior_v1"
SELECTED_MODEL_FAMILY = "timevae"
SELECTED_SYNTHETIC_METHOD = "prior_v1"

SYNTHETIC_METHOD_CONFIGS = {
    "kovae": {
        "base_dir": PROJECT_ROOT / "data/synthetic_subjects/kovae",
        "methods": ["rollout_v1", "posterior_bank_v2"],
    },
    "timevae": {
        "base_dir": PROJECT_ROOT / "data/synthetic_subjects/timevae",
        "methods": ["prior_v1"],
    },
}

if SELECTED_MODEL_FAMILY not in SYNTHETIC_METHOD_CONFIGS:
    raise ValueError(
        f"Unknown SELECTED_MODEL_FAMILY={SELECTED_MODEL_FAMILY}. "
        f"Available: {list(SYNTHETIC_METHOD_CONFIGS.keys())}"
    )

if SELECTED_SYNTHETIC_METHOD not in SYNTHETIC_METHOD_CONFIGS[SELECTED_MODEL_FAMILY]["methods"]:
    raise ValueError(
        f"Unknown SELECTED_SYNTHETIC_METHOD={SELECTED_SYNTHETIC_METHOD} for {SELECTED_MODEL_FAMILY}. "
        f"Available: {SYNTHETIC_METHOD_CONFIGS[SELECTED_MODEL_FAMILY]['methods']}"
    )

OUT_DIR = (
    PROJECT_ROOT
    / "results/downstream_cnnbilstm"
    / SELECTED_MODEL_FAMILY
    / SELECTED_SYNTHETIC_METHOD
)

REAL_DIR = PROJECT_ROOT / "data/processed/native_rates"
SYN_DIR = SYNTHETIC_METHOD_CONFIGS[SELECTED_MODEL_FAMILY]["base_dir"] / SELECTED_SYNTHETIC_METHOD

# Put your friend's unzipped pretrained folder here:
# /home/iailab42/khans1/projects/ir/models/downstream/pretrained/cnnbilstm_native_results/
PRETRAINED_RESULTS_DIR = (
    PROJECT_ROOT
    / "models/downstream/pretrained/cnnbilstm_native_results"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Training settings. These are close to your original CNN-BiLSTM script.
BATCH_SIZE = 384
NUM_WORKERS = 8
EPOCHS_CNNBILSTM = 100
LR = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.5
PATIENCE = 25

USE_CLASS_WEIGHTS = True
USE_AMP = True
USE_TF32 = True
SAVE_CONFUSION_MATRIX_PNG = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# =========================
# PATHS AND CONSTANTS
# =========================

REAL_X_ACC_PATH = REAL_DIR / "all_X_acc_32hz.npy"
REAL_X_BVP_PATH = REAL_DIR / "all_X_bvp_64hz.npy"
REAL_X_SLOW_PATH = REAL_DIR / "all_X_slow_4hz.npy"
REAL_Y_PATH = REAL_DIR / "all_y.npy"
REAL_SUBJECT_PATH = REAL_DIR / "all_subject.npy"

SYN_X_ACC_PATH = SYN_DIR / "generated_subjects_X_acc_32hz.npy"
SYN_X_BVP_PATH = SYN_DIR / "generated_subjects_X_bvp_64hz.npy"
SYN_X_SLOW_PATH = SYN_DIR / "generated_subjects_X_slow_4hz.npy"
SYN_Y_PATH = SYN_DIR / "generated_subjects_all_y.npy"
SYN_SUBJECT_PATH = SYN_DIR / "generated_subjects_all_subject.npy"

ACTIVITY_IDS = [1, 2, 3, 4, 5, 6, 7, 8]
NUM_CLASSES = len(ACTIVITY_IDS)
LABEL_TO_INDEX = {label: i for i, label in enumerate(ACTIVITY_IDS)}
INDEX_TO_LABEL = {i: label for label, i in LABEL_TO_INDEX.items()}

TRAIN_SUBJECTS = ["S1", "S2", "S3", "S4", "S5", "S6", "S9", "S11", "S12", "S13"]
VAL_SUBJECTS = ["S14", "S15"]
TEST_SUBJECTS = ["S7", "S8", "S10"]

# Four subsequences per modality, same as your original CNN-BiLSTM code:
#   ACC:  256 = 4 x 64
#   BVP:  512 = 4 x 128
#   EDA:   32 = 4 x 8
#   TEMP:  32 = 4 x 8
N_SEQ = 4

BRANCH_SPECS = {
    "acc": {
        "real_path": REAL_X_ACC_PATH,
        "syn_path": SYN_X_ACC_PATH,
        "expected_shape_tail": (256, 3),
        "seq_len": 256,
        "in_channels": 3,
        "native_hz": 32,
        "channel_names": ["ACC_x", "ACC_y", "ACC_z"],
        "description": "ACC only",
        "display_name": "ACC",
    },
    "bvp": {
        "real_path": REAL_X_BVP_PATH,
        "syn_path": SYN_X_BVP_PATH,
        "expected_shape_tail": (512, 1),
        "seq_len": 512,
        "in_channels": 1,
        "native_hz": 64,
        "channel_names": ["BVP"],
        "description": "BVP only",
        "display_name": "BVP",
    },
    "slow": {
        "real_path": REAL_X_SLOW_PATH,
        "syn_path": SYN_X_SLOW_PATH,
        "expected_shape_tail": (32, 2),
        "seq_len": 32,
        "in_channels": 2,
        "native_hz": 4,
        "channel_names": ["EDA", "TEMP"],
        "description": "EDA+TEMP slow pair",
        "display_name": "SLOW",
    },
    "eda": {
        "expected_shape_tail": (32, 1),
        "seq_len": 32,
        "in_channels": 1,
        "native_hz": 4,
        "channel_names": ["EDA"],
        "description": "EDA only",
        "display_name": "EDA",
    },
    "temp": {
        "expected_shape_tail": (32, 1),
        "seq_len": 32,
        "in_channels": 1,
        "native_hz": 4,
        "channel_names": ["TEMP"],
        "description": "TEMP only",
        "display_name": "TEMP",
    },
}

FUSED_MODALITY_NAME = "fused"
FUSED_BRANCHES = ["acc", "bvp", "eda", "temp"]
FUSED_NATIVE_HZ = "32/64/4/4"
FUSED_CHANNEL_NAMES = ["ACC_x", "ACC_y", "ACC_z", "BVP", "EDA", "TEMP"]


# =========================
# CUDA BACKEND SETTINGS
# =========================

if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = USE_TF32
    torch.backends.cudnn.allow_tf32 = USE_TF32
    torch.backends.cudnn.benchmark = False


# =========================
# HELPERS
# =========================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return path


def subject_sort_key(s):
    s = str(s)
    if s.startswith("S") and s[1:].isdigit():
        return ("S", int(s[1:]))
    m = re.search(r"(\d+)$", s)
    if m is not None:
        return (s[:m.start(1)], int(m.group(1)))
    return (s, -1)


def labels_to_indices(y):
    y = np.asarray(y).astype(np.int64)
    out = np.zeros_like(y, dtype=np.int64)
    for i, lab in enumerate(y):
        lab = int(lab)
        if lab not in LABEL_TO_INDEX:
            raise ValueError(f"Unexpected label {lab}. Expected labels: {ACTIVITY_IDS}")
        out[i] = LABEL_TO_INDEX[lab]
    return out


def indices_to_labels(y_idx):
    y_idx = np.asarray(y_idx).astype(np.int64)
    return np.array([INDEX_TO_LABEL[int(i)] for i in y_idx], dtype=np.int64)


def split_slow_to_eda_temp(X_slow):
    X_slow = np.asarray(X_slow, dtype=np.float32)
    if X_slow.ndim != 3 or X_slow.shape[1:] != (32, 2):
        raise ValueError(f"Expected SLOW [N,32,2], got {X_slow.shape}")
    return X_slow[:, :, 0:1].astype(np.float32), X_slow[:, :, 1:2].astype(np.float32)


def filter_valid_activities(X_dict, y, subjects):
    keep = np.isin(y, np.array(ACTIVITY_IDS, dtype=np.int64))
    return {k: v[keep] for k, v in X_dict.items()}, y[keep].astype(np.int64), subjects[keep].astype(str)


def filter_by_subjects(X_dict, y, subjects, selected_subjects):
    selected_subjects = set(str(s) for s in selected_subjects)
    keep = np.array([str(s) in selected_subjects for s in subjects], dtype=bool)
    return {k: v[keep] for k, v in X_dict.items()}, y[keep].astype(np.int64), subjects[keep].astype(str)


def sanitize_name(name):
    return str(name).replace(" ", "_").replace("/", "_").replace("\\", "_")


def safe_display(df, max_rows=20):
    try:
        display(df)
    except Exception:
        print(df.head(max_rows))


def autocast_context():
    if DEVICE == "cuda":
        return torch.amp.autocast("cuda", enabled=USE_AMP)
    return nullcontext()


def make_grad_scaler():
    if DEVICE != "cuda":
        return torch.amp.GradScaler("cuda", enabled=False)
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except TypeError:
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)


def shape_dict(X_dict):
    return {k: list(np.asarray(v).shape) for k, v in X_dict.items()}


def shape_dict_string(X_dict):
    return json.dumps(shape_dict(X_dict), sort_keys=True)


def channel_string(channels):
    return ",".join(channels)


# =========================
# METRICS AND SAVING
# =========================

def compute_classification_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)

    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_precision": float(precision_score(y_true, y_pred, labels=ACTIVITY_IDS, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, labels=ACTIVITY_IDS, average="macro", zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=ACTIVITY_IDS, average="macro", zero_division=0)),
        "weighted_precision": float(precision_score(y_true, y_pred, labels=ACTIVITY_IDS, average="weighted", zero_division=0)),
        "weighted_recall": float(recall_score(y_true, y_pred, labels=ACTIVITY_IDS, average="weighted", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, labels=ACTIVITY_IDS, average="weighted", zero_division=0)),
    }
    metrics["balanced_accuracy"] = metrics["macro_recall"]
    return metrics


def compute_per_activity_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=ACTIVITY_IDS, zero_division=0
    )

    rows = []
    for i, activity in enumerate(ACTIVITY_IDS):
        true_mask = y_true == activity
        correct = int(np.sum(true_mask & (y_pred == activity)))
        total = int(np.sum(true_mask))
        rows.append({
            "activity_label": int(activity),
            "precision": float(precision[i]),
            "recall": float(recall[i]),
            "f1": float(f1[i]),
            "support": int(support[i]),
            "correct_true_activity_windows": correct,
            "total_true_activity_windows": total,
            "true_activity_window_accuracy": float(correct / total) if total > 0 else np.nan,
        })
    return pd.DataFrame(rows)


def plot_confusion_matrix_image(cm, labels, title, out_path, normalize=False):
    cm_to_plot = cm.astype(np.float64)
    if normalize:
        row_sum = cm_to_plot.sum(axis=1, keepdims=True)
        cm_to_plot = np.divide(cm_to_plot, row_sum, out=np.zeros_like(cm_to_plot), where=row_sum != 0)

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(cm_to_plot, cmap="Blues")
    ax.figure.colorbar(im, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted activity")
    ax.set_ylabel("True activity")
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)

    threshold = cm_to_plot.max() / 2.0 if cm_to_plot.size and cm_to_plot.max() > 0 else 0.0
    for i in range(cm_to_plot.shape[0]):
        for j in range(cm_to_plot.shape[1]):
            text = f"{cm_to_plot[i, j]:.2f}" if normalize else str(int(cm[i, j]))
            ax.text(
                j,
                i,
                text,
                ha="center",
                va="center",
                color="white" if cm_to_plot[i, j] > threshold else "black",
            )

    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def save_predictions_cm_and_activity_report(
    framework,
    model_name,
    experiment,
    modality,
    native_hz,
    channels,
    y_true,
    y_pred,
    y_prob=None,
):
    safe_name = sanitize_name(f"{framework}_{experiment}_{modality}")
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)

    pred_df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred, "correct": y_true == y_pred})
    if y_prob is not None:
        y_prob = np.asarray(y_prob, dtype=np.float32)
        for i, activity in enumerate(ACTIVITY_IDS):
            pred_df[f"prob_activity_{activity}"] = y_prob[:, i]

    pred_path = OUT_DIR / f"{safe_name}_predictions.csv"
    pred_df.to_csv(pred_path, index=False)

    cm = confusion_matrix(y_true, y_pred, labels=ACTIVITY_IDS)
    cm_df = pd.DataFrame(cm, index=[f"true_{a}" for a in ACTIVITY_IDS], columns=[f"pred_{a}" for a in ACTIVITY_IDS])
    cm_csv_path = OUT_DIR / f"{safe_name}_confusion_matrix.csv"
    cm_df.to_csv(cm_csv_path)

    cm_count_png = OUT_DIR / f"{safe_name}_confusion_matrix_counts.png"
    cm_norm_png = OUT_DIR / f"{safe_name}_confusion_matrix_normalized.png"
    if SAVE_CONFUSION_MATRIX_PNG:
        plot_confusion_matrix_image(cm, ACTIVITY_IDS, f"{framework} {experiment} {modality} confusion matrix", cm_count_png, normalize=False)
        plot_confusion_matrix_image(cm, ACTIVITY_IDS, f"{framework} {experiment} {modality} normalized confusion matrix", cm_norm_png, normalize=True)

    activity_df = compute_per_activity_metrics(y_true, y_pred)
    activity_df.insert(0, "framework", framework)
    activity_df.insert(1, "model", model_name)
    activity_df.insert(2, "experiment", experiment)
    activity_df.insert(3, "native_modality", modality)
    activity_df.insert(4, "native_hz", native_hz)
    activity_df.insert(5, "channels", channel_string(channels))

    activity_path = OUT_DIR / f"{safe_name}_per_activity_metrics.csv"
    activity_df.to_csv(activity_path, index=False)

    print("Saved:", pred_path)
    print("Saved:", cm_csv_path)
    if SAVE_CONFUSION_MATRIX_PNG:
        print("Saved:", cm_count_png)
        print("Saved:", cm_norm_png)
    print("Saved:", activity_path)
    return activity_df


# =========================
# DATASET VIEWS
# =========================

def load_native_data(load_synthetic):
    require_file(REAL_Y_PATH)
    require_file(REAL_SUBJECT_PATH)

    real_y = np.load(REAL_Y_PATH).astype(np.int64)
    real_subjects = np.load(REAL_SUBJECT_PATH, allow_pickle=True).astype(str)
    if len(real_y) != len(real_subjects):
        raise ValueError(f"Real y/subject mismatch: {len(real_y)} vs {len(real_subjects)}")

    real_X = {}
    for modality_name in ["acc", "bvp", "slow"]:
        info = BRANCH_SPECS[modality_name]
        require_file(info["real_path"])
        Xr = np.load(info["real_path"]).astype(np.float32)
        if Xr.ndim != 3 or Xr.shape[1:] != info["expected_shape_tail"]:
            raise ValueError(f"Real {modality_name}: expected [N,{info['expected_shape_tail']}], got {Xr.shape}")
        if len(Xr) != len(real_y):
            raise ValueError(f"Real {modality_name}/y mismatch: {len(Xr)} vs {len(real_y)}")
        real_X[modality_name] = Xr

    real_X, real_y, real_subjects = filter_valid_activities(real_X, real_y, real_subjects)
    print("\nLoaded real data:")
    print("  y:", real_y.shape)
    print("  subjects:", real_subjects.shape)
    for k, v in real_X.items():
        print(f"  {k}: {v.shape}")

    if not load_synthetic:
        print("\nSynthetic data skipped because RUN_REAL_TO_REAL_ONLY=True.")
        return real_X, real_y, real_subjects, None, None, None

    require_file(SYN_Y_PATH)
    require_file(SYN_SUBJECT_PATH)
    syn_y = np.load(SYN_Y_PATH).astype(np.int64)
    syn_subjects = np.load(SYN_SUBJECT_PATH, allow_pickle=True).astype(str)
    if len(syn_y) != len(syn_subjects):
        raise ValueError(f"Synthetic y/subject mismatch: {len(syn_y)} vs {len(syn_subjects)}")

    syn_X = {}
    for modality_name in ["acc", "bvp", "slow"]:
        info = BRANCH_SPECS[modality_name]
        require_file(info["syn_path"])
        Xs = np.load(info["syn_path"]).astype(np.float32)
        if Xs.ndim != 3 or Xs.shape[1:] != info["expected_shape_tail"]:
            raise ValueError(f"Synthetic {modality_name}: expected [N,{info['expected_shape_tail']}], got {Xs.shape}")
        if len(Xs) != len(syn_y):
            raise ValueError(f"Synthetic {modality_name}/y mismatch: {len(Xs)} vs {len(syn_y)}")
        syn_X[modality_name] = Xs

    syn_X, syn_y, syn_subjects = filter_valid_activities(syn_X, syn_y, syn_subjects)
    print("\nLoaded synthetic data:")
    print("  y:", syn_y.shape)
    print("  subjects:", syn_subjects.shape)
    for k, v in syn_X.items():
        print(f"  {k}: {v.shape}")

    return real_X, real_y, real_subjects, syn_X, syn_y, syn_subjects


def validate_fixed_split(real_subjects):
    available = sorted(np.unique(real_subjects.astype(str)), key=subject_sort_key)
    missing_train = sorted(set(TRAIN_SUBJECTS) - set(available), key=subject_sort_key)
    missing_val = sorted(set(VAL_SUBJECTS) - set(available), key=subject_sort_key)
    missing_test = sorted(set(TEST_SUBJECTS) - set(available), key=subject_sort_key)

    if missing_train or missing_val or missing_test:
        raise ValueError(
            "Fixed split subjects missing from real data.\n"
            f"Missing train: {missing_train}\nMissing val: {missing_val}\nMissing test: {missing_test}\nAvailable: {available}"
        )
    if set(TRAIN_SUBJECTS) & set(VAL_SUBJECTS) or set(TRAIN_SUBJECTS) & set(TEST_SUBJECTS) or set(VAL_SUBJECTS) & set(TEST_SUBJECTS):
        raise ValueError("Train/val/test subject split overlaps.")

    print("\nFixed split OK:")
    print("  Train:", TRAIN_SUBJECTS)
    print("  Val:  ", VAL_SUBJECTS)
    print("  Test: ", TEST_SUBJECTS)


def make_single_view(branch_name, real_train_eval, real_val_eval, real_test_eval,
                     real_train_y, real_val_y, real_test_y,
                     syn_eval=None, syn_test_eval=None, syn_y_all=None, syn_test_y=None,
                     syn_test_subjects=None):
    spec = BRANCH_SPECS[branch_name]
    item = {
        "branch_names": [branch_name],
        "real_train_X_native": {branch_name: real_train_eval[branch_name]},
        "real_val_X_native": {branch_name: real_val_eval[branch_name]},
        "real_test_X_native": {branch_name: real_test_eval[branch_name]},
        "real_train_y": real_train_y,
        "real_val_y": real_val_y,
        "real_test_y": real_test_y,
        "native_hz": spec["native_hz"],
        "channels": spec["channel_names"],
        "description": spec["description"],
    }
    if syn_eval is not None:
        item.update({
            "syn_X_native": {branch_name: syn_eval[branch_name]},
            "syn_test_X_native": {branch_name: syn_test_eval[branch_name]},
            "syn_y": syn_y_all,
            "syn_test_y": syn_test_y,
            "syn_test_subjects": syn_test_subjects,
        })
    return item


def make_fused_view(real_train_eval, real_val_eval, real_test_eval,
                    real_train_y, real_val_y, real_test_y,
                    syn_eval=None, syn_test_eval=None, syn_y_all=None, syn_test_y=None,
                    syn_test_subjects=None):
    item = {
        "branch_names": list(FUSED_BRANCHES),
        "real_train_X_native": {k: real_train_eval[k] for k in FUSED_BRANCHES},
        "real_val_X_native": {k: real_val_eval[k] for k in FUSED_BRANCHES},
        "real_test_X_native": {k: real_test_eval[k] for k in FUSED_BRANCHES},
        "real_train_y": real_train_y,
        "real_val_y": real_val_y,
        "real_test_y": real_test_y,
        "native_hz": FUSED_NATIVE_HZ,
        "channels": FUSED_CHANNEL_NAMES,
        "description": "ACC+BVP+EDA+TEMP fused native-rate multibranch",
    }
    if syn_eval is not None:
        item.update({
            "syn_X_native": {k: syn_eval[k] for k in FUSED_BRANCHES},
            "syn_test_X_native": {k: syn_test_eval[k] for k in FUSED_BRANCHES},
            "syn_y": syn_y_all,
            "syn_test_y": syn_test_y,
            "syn_test_subjects": syn_test_subjects,
        })
    return item


def make_native_dataset_views(real_X_all, real_y_all, real_subjects_all, syn_X_all=None, syn_y_all=None, syn_subjects_all=None):
    validate_fixed_split(real_subjects_all)
    use_synthetic = syn_X_all is not None and syn_y_all is not None and syn_subjects_all is not None

    real_train_X, real_train_y, _ = filter_by_subjects(real_X_all, real_y_all, real_subjects_all, TRAIN_SUBJECTS)
    real_val_X, real_val_y, _ = filter_by_subjects(real_X_all, real_y_all, real_subjects_all, VAL_SUBJECTS)
    real_test_X, real_test_y, _ = filter_by_subjects(real_X_all, real_y_all, real_subjects_all, TEST_SUBJECTS)

    real_train_eda, real_train_temp = split_slow_to_eda_temp(real_train_X["slow"])
    real_val_eda, real_val_temp = split_slow_to_eda_temp(real_val_X["slow"])
    real_test_eda, real_test_temp = split_slow_to_eda_temp(real_test_X["slow"])

    real_train_eval = {"acc": real_train_X["acc"], "bvp": real_train_X["bvp"], "eda": real_train_eda, "temp": real_train_temp}
    real_val_eval = {"acc": real_val_X["acc"], "bvp": real_val_X["bvp"], "eda": real_val_eda, "temp": real_val_temp}
    real_test_eval = {"acc": real_test_X["acc"], "bvp": real_test_X["bvp"], "eda": real_test_eda, "temp": real_test_temp}

    syn_eval = None
    syn_test_eval = None
    syn_test_subjects = []
    syn_test_y = None

    if use_synthetic:
        unique_syn_subjects = sorted(np.unique(syn_subjects_all.astype(str)), key=subject_sort_key)
        if len(unique_syn_subjects) < SYN_TEST_N_SUBJECTS:
            raise ValueError(f"Need at least {SYN_TEST_N_SUBJECTS} synthetic subjects, found {len(unique_syn_subjects)}")
        syn_test_subjects = unique_syn_subjects[:SYN_TEST_N_SUBJECTS]
        syn_test_X, syn_test_y, _ = filter_by_subjects(syn_X_all, syn_y_all, syn_subjects_all, syn_test_subjects)
        print("\nSynthetic test subjects:", syn_test_subjects)

        syn_eda, syn_temp = split_slow_to_eda_temp(syn_X_all["slow"])
        syn_test_eda, syn_test_temp = split_slow_to_eda_temp(syn_test_X["slow"])
        syn_eval = {"acc": syn_X_all["acc"], "bvp": syn_X_all["bvp"], "eda": syn_eda, "temp": syn_temp}
        syn_test_eval = {"acc": syn_test_X["acc"], "bvp": syn_test_X["bvp"], "eda": syn_test_eda, "temp": syn_test_temp}

    views = {}
    for modality in ["acc", "bvp", "eda", "temp"]:
        views[modality] = make_single_view(
            modality,
            real_train_eval,
            real_val_eval,
            real_test_eval,
            real_train_y,
            real_val_y,
            real_test_y,
            syn_eval=syn_eval,
            syn_test_eval=syn_test_eval,
            syn_y_all=syn_y_all,
            syn_test_y=syn_test_y,
            syn_test_subjects=syn_test_subjects,
        )

    views[FUSED_MODALITY_NAME] = make_fused_view(
        real_train_eval,
        real_val_eval,
        real_test_eval,
        real_train_y,
        real_val_y,
        real_test_y,
        syn_eval=syn_eval,
        syn_test_eval=syn_test_eval,
        syn_y_all=syn_y_all,
        syn_test_y=syn_test_y,
        syn_test_subjects=syn_test_subjects,
    )

    for name, view in views.items():
        msg = (
            f"  View {name:6s}: "
            f"train={shape_dict(view['real_train_X_native'])}, "
            f"val={shape_dict(view['real_val_X_native'])}, "
            f"test={shape_dict(view['real_test_X_native'])}"
        )
        if use_synthetic:
            msg += (
                f", syn_all={shape_dict(view['syn_X_native'])}, "
                f"syn_test3={shape_dict(view['syn_test_X_native'])}"
            )
        print(msg)

    return views


def save_shape_report(views):
    rows = []
    for name, view in views.items():
        row = {
            "native_view": name,
            "native_hz": view["native_hz"],
            "channels": channel_string(view["channels"]),
            "branches": ",".join(view["branch_names"]),
            "description": view.get("description", ""),
            "native_real_train_shape_by_branch_N_T_C": shape_dict_string(view["real_train_X_native"]),
            "native_real_val_shape_by_branch_N_T_C": shape_dict_string(view["real_val_X_native"]),
            "native_real_test_shape_by_branch_N_T_C": shape_dict_string(view["real_test_X_native"]),
            "framework_input_style": "dict_of_native_rate_N_T_C_tensors",
        }
        if "syn_X_native" in view:
            row.update({
                "native_syn_all_shape_by_branch_N_T_C": shape_dict_string(view["syn_X_native"]),
                "native_syn_test3_shape_by_branch_N_T_C": shape_dict_string(view["syn_test_X_native"]),
                "syn_test_subjects": ",".join(view.get("syn_test_subjects", [])),
            })
        rows.append(row)

    df = pd.DataFrame(rows)
    out_path = OUT_DIR / "native_input_shape_report.csv"
    df.to_csv(out_path, index=False)
    print("\nSaved shape report:", out_path)
    return df


# =========================
# DATASET / DATALOADER
# =========================

class NativeMultiInputDataset(Dataset):
    def __init__(self, X_dict, y_original, branch_names):
        self.branch_names = list(branch_names)
        self.X = {}
        n = None

        for branch in self.branch_names:
            if branch not in BRANCH_SPECS:
                raise ValueError(f"Unknown branch: {branch}")
            spec = BRANCH_SPECS[branch]
            Xb = np.asarray(X_dict[branch], dtype=np.float32)
            expected_tail = spec["expected_shape_tail"]
            if Xb.ndim != 3 or Xb.shape[1:] != expected_tail:
                raise ValueError(f"{branch}: expected [N,{expected_tail}], got {Xb.shape}")
            if n is None:
                n = len(Xb)
            elif len(Xb) != n:
                raise ValueError(f"Branch length mismatch: {branch} has {len(Xb)}, expected {n}")
            self.X[branch] = torch.from_numpy(Xb)

        y_original = np.asarray(y_original, dtype=np.int64)
        if n is None:
            raise ValueError("No input branches provided.")
        if len(y_original) != n:
            raise ValueError(f"X/y mismatch: X={n}, y={len(y_original)}")

        self.y = torch.from_numpy(labels_to_indices(y_original)).long()
        self.y_original = y_original

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {
            "x": {branch: self.X[branch][idx] for branch in self.branch_names},
            "y": self.y[idx],
        }


def make_loader(X_dict, y, branch_names, batch_size, shuffle):
    dataset = NativeMultiInputDataset(X_dict, y, branch_names)
    drop_last = bool(shuffle and len(dataset) > batch_size)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        drop_last=drop_last,
        persistent_workers=(NUM_WORKERS > 0),
    )
    return loader


def compute_class_weights(y_original):
    y_idx = labels_to_indices(y_original)
    counts = np.bincount(y_idx, minlength=NUM_CLASSES).astype(np.float32)
    counts = np.maximum(counts, 1.0)
    weights = counts.sum() / (NUM_CLASSES * counts)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)


def move_batch_to_device(batch):
    x = {k: v.to(DEVICE, non_blocking=True) for k, v in batch["x"].items()}
    y = batch["y"].to(DEVICE, non_blocking=True)
    return x, y


# =========================
# CNN-BiLSTM MODEL
# =========================

class TimeDistributedConvBranch(nn.Module):
    def __init__(self, in_channels, kernel_size, dropout):
        super().__init__()
        padding = kernel_size // 2
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=kernel_size, padding=padding),
            nn.ReLU(inplace=True),
            nn.Conv1d(64, 32, kernel_size=kernel_size, padding=padding),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.MaxPool1d(kernel_size=2),
        )

    def forward(self, x):
        # x: [B, S, steps, C]
        B, S, steps, C = x.shape
        x = x.reshape(B * S, steps, C).transpose(1, 2)  # [B*S, C, steps]
        h = self.net(x)
        h = h.reshape(B, S, -1)
        return h


class ModalityCNNBiLSTMEncoder(nn.Module):
    def __init__(self, branch_name, dropout):
        super().__init__()
        spec = BRANCH_SPECS[branch_name]
        seq_len = int(spec["seq_len"])
        in_channels = int(spec["in_channels"])
        name = str(spec["display_name"])

        if seq_len % N_SEQ != 0:
            raise ValueError(f"{name}: seq_len={seq_len} not divisible by N_SEQ={N_SEQ}")

        self.branch_name = branch_name
        self.in_channels = in_channels
        self.seq_len = seq_len
        self.n_seq = N_SEQ
        self.n_steps = seq_len // N_SEQ
        self.name = name

        self.branch_k3 = TimeDistributedConvBranch(in_channels, kernel_size=3, dropout=dropout)
        self.branch_k7 = TimeDistributedConvBranch(in_channels, kernel_size=7, dropout=dropout)
        self.branch_k11 = TimeDistributedConvBranch(in_channels, kernel_size=11, dropout=dropout)

        branch_out_dim = 32 * (self.n_steps // 2)
        lstm_input_dim = branch_out_dim * 3

        self.bilstm1 = nn.LSTM(
            input_size=lstm_input_dim,
            hidden_size=64,
            batch_first=True,
            bidirectional=True,
        )
        self.bilstm2 = nn.LSTM(
            input_size=64 * 2,
            hidden_size=32,
            batch_first=True,
            bidirectional=True,
        )
        self.output_dim = 32 * 2

    def forward(self, x):
        # x: [B, T, C]
        B, T, C = x.shape
        if T != self.seq_len:
            raise ValueError(f"{self.name}: expected T={self.seq_len}, got T={T}")
        if C != self.in_channels:
            raise ValueError(f"{self.name}: expected C={self.in_channels}, got C={C}")

        x = x.reshape(B, self.n_seq, self.n_steps, C)

        h3 = self.branch_k3(x)
        h7 = self.branch_k7(x)
        h11 = self.branch_k11(x)
        h = torch.cat([h3, h7, h11], dim=-1)

        h, _ = self.bilstm1(h)
        h, (h_n, _) = self.bilstm2(h)
        h_final = torch.cat([h_n[-2], h_n[-1]], dim=1)
        return h_final


class NativeRateCNNBiLSTM(nn.Module):
    def __init__(self, branch_names, num_classes, dropout):
        super().__init__()
        self.branch_names = list(branch_names)
        if not self.branch_names:
            raise ValueError("At least one branch must be enabled.")

        self.encoders = nn.ModuleDict()
        fusion_dim = 0
        for branch in self.branch_names:
            encoder = ModalityCNNBiLSTMEncoder(branch, dropout=dropout)
            self.encoders[branch] = encoder
            fusion_dim += encoder.output_dim

        self.fc = nn.Linear(fusion_dim, 128)
        self.bn = nn.BatchNorm1d(128)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x_dict):
        features = []
        for branch in self.branch_names:
            features.append(self.encoders[branch](x_dict[branch]))
        h = torch.cat(features, dim=1)
        z = self.fc(h)
        z = self.bn(z)
        z = F.relu(z)
        z = self.dropout(z)
        logits = self.classifier(z)
        return logits


def make_new_model(branch_names):
    return NativeRateCNNBiLSTM(
        branch_names=branch_names,
        num_classes=NUM_CLASSES,
        dropout=DROPOUT,
    )


# =========================
# EVALUATION
# =========================

def evaluate_model(model, loader):
    model.eval()
    all_true = []
    all_pred = []
    all_prob = []

    with torch.no_grad():
        for batch in loader:
            x_dict, y_batch = move_batch_to_device(batch)
            with autocast_context():
                logits = model(x_dict)
            probs = torch.softmax(logits.float(), dim=1)
            pred = torch.argmax(probs, dim=1)

            all_true.append(y_batch.detach().cpu().numpy())
            all_pred.append(pred.detach().cpu().numpy())
            all_prob.append(probs.detach().cpu().numpy())

    y_true_idx = np.concatenate(all_true)
    y_pred_idx = np.concatenate(all_pred)
    y_prob = np.concatenate(all_prob)
    y_true = indices_to_labels(y_true_idx)
    y_pred = indices_to_labels(y_pred_idx)
    return y_true, y_pred, y_prob


# =========================
# EXPERIMENTS AND SAVED ROWS
# =========================

def get_experiments_for_view(view):
    real_to_real = {
        "name": "real_to_real",
        "train_X": view["real_train_X_native"],
        "train_y": view["real_train_y"],
        "val_X": view["real_val_X_native"],
        "val_y": view["real_val_y"],
        "test_X": view["real_test_X_native"],
        "test_y": view["real_test_y"],
    }

    if RUN_REAL_TO_REAL_ONLY:
        return [real_to_real]

    missing_syn_keys = [k for k in ["syn_X_native", "syn_test_X_native", "syn_y", "syn_test_y"] if k not in view]
    if missing_syn_keys:
        raise ValueError(f"Full evaluation needs synthetic data. Missing keys: {missing_syn_keys}")

    return [
        real_to_real,
        {
            "name": "real_to_synthetic",
            "train_X": view["real_train_X_native"],
            "train_y": view["real_train_y"],
            "val_X": view["real_val_X_native"],
            "val_y": view["real_val_y"],
            "test_X": view["syn_test_X_native"],
            "test_y": view["syn_test_y"],
        },
        {
            "name": "synthetic_to_real",
            "train_X": view["syn_X_native"],
            "train_y": view["syn_y"],
            "val_X": view["real_val_X_native"],
            "val_y": view["real_val_y"],
            "test_X": view["real_test_X_native"],
            "test_y": view["real_test_y"],
        },
        {
            "name": "real_plus_synthetic_to_real",
            "train_X": {k: np.concatenate([view["real_train_X_native"][k], view["syn_X_native"][k]], axis=0)
                        for k in view["branch_names"]},
            "train_y": np.concatenate([view["real_train_y"], view["syn_y"]], axis=0),
            "val_X": view["real_val_X_native"],
            "val_y": view["real_val_y"],
            "test_X": view["real_test_X_native"],
            "test_y": view["real_test_y"],
        },
    ]


def result_csv_path(framework="cnnbilstm"):
    return OUT_DIR / f"{framework}_native_downstream_results.csv"


def activity_csv_path(framework="cnnbilstm"):
    return OUT_DIR / f"{framework}_native_per_activity_results.csv"



def _candidate_existing_result_csv_paths(framework):
    paths = [
        result_csv_path(framework),
    ]

    if PRETRAINED_RESULTS_DIR.exists():
        paths.extend([
            PRETRAINED_RESULTS_DIR / f"{framework}_native_downstream_results.csv",
            PRETRAINED_RESULTS_DIR / "all_framework_native_results.csv",
        ])

    unique = []
    seen = set()
    for path in paths:
        path = Path(path)
        key = str(path)
        if key not in seen:
            unique.append(path)
            seen.add(key)
    return unique


def _candidate_existing_activity_csv_paths(framework):
    paths = [
        activity_csv_path(framework),
    ]

    if PRETRAINED_RESULTS_DIR.exists():
        paths.extend([
            PRETRAINED_RESULTS_DIR / f"{framework}_native_per_activity_results.csv",
            PRETRAINED_RESULTS_DIR / "all_framework_native_per_activity_results.csv",
        ])

    unique = []
    seen = set()
    for path in paths:
        path = Path(path)
        key = str(path)
        if key not in seen:
            unique.append(path)
            seen.add(key)
    return unique


def _resolve_pretrained_path(path_like):
    if str(path_like).lower() in {"", "nan", "none"}:
        return None

    p = Path(path_like)

    candidates = [
        p,
        PROJECT_ROOT / p,
        PRETRAINED_RESULTS_DIR / p.name,
        PRETRAINED_RESULTS_DIR / "cnnbilstm_saved_models" / p.name,
        PRETRAINED_RESULTS_DIR.parent / p,
    ]

    for candidate in candidates:
        if Path(candidate).exists():
            return Path(candidate)

    return None


def get_existing_result_row(framework, experiment, modality, model=None):
    for path in _candidate_existing_result_csv_paths(framework):
        if not path.exists():
            continue
        try:
            df = pd.read_csv(path)
        except Exception:
            continue

        required = {"framework", "experiment", "native_modality"}
        if not required.issubset(df.columns):
            continue

        mask = (
            (df["framework"].astype(str) == framework)
            & (df["experiment"].astype(str) == experiment)
            & (df["native_modality"].astype(str) == modality)
        )
        if model is not None and "model" in df.columns:
            mask &= df["model"].astype(str) == str(model)

        if mask.any():
            row = df.loc[mask].iloc[-1].to_dict()

            # If this row came from the pretrained folder, resolve saved paths locally.
            saved_model = row.get("saved_best_model_file", "")
            resolved_model = _resolve_pretrained_path(saved_model)
            if resolved_model is not None:
                row["saved_best_model_file"] = str(resolved_model)

            history_file = row.get("training_history_file", "")
            resolved_history = _resolve_pretrained_path(history_file)
            if resolved_history is not None:
                row["training_history_file"] = str(resolved_history)

            return row

    return None


def get_existing_activity_rows(framework, experiment, modality, model=None):
    for path in _candidate_existing_activity_csv_paths(framework):
        if not path.exists():
            continue
        try:
            df = pd.read_csv(path)
        except Exception:
            continue

        required = {"framework", "experiment", "native_modality"}
        if not required.issubset(df.columns):
            continue

        mask = (
            (df["framework"].astype(str) == framework)
            & (df["experiment"].astype(str) == experiment)
            & (df["native_modality"].astype(str) == modality)
        )
        if model is not None and "model" in df.columns:
            mask &= df["model"].astype(str) == str(model)

        if mask.any():
            return df.loc[mask].copy()

    return pd.DataFrame()


def get_cnnbilstm_model_file(experiment, modality):
    model_dir = OUT_DIR / "cnnbilstm_saved_models"
    stem = sanitize_name(f"best_{CNN_MODEL_NAME}_{experiment}_{modality}")
    return model_dir / f"{stem}.pt"


def find_cnnbilstm_real_model_file(modality):
    candidates = []
    row = get_existing_result_row("cnnbilstm", "real_to_real", modality, model=CNN_MODEL_NAME)
    if row is not None:
        saved = row.get("saved_best_model_file", "")
        resolved = _resolve_pretrained_path(saved)
        if resolved is not None:
            candidates.append(resolved)
        elif str(saved).lower() not in {"", "nan", "none"}:
            candidates.append(Path(saved))

    candidates.append(get_cnnbilstm_model_file("real_to_real", modality))

    model_dir = OUT_DIR / "cnnbilstm_saved_models"
    if model_dir.exists():
        candidates.extend(sorted(model_dir.glob(f"best_{CNN_MODEL_NAME}_real_to_real_{sanitize_name(modality)}.pt")))

    pretrained_model_dir = PRETRAINED_RESULTS_DIR / "cnnbilstm_saved_models"
    if pretrained_model_dir.exists():
        candidates.extend(sorted(pretrained_model_dir.glob(f"best_{CNN_MODEL_NAME}_real_to_real_{sanitize_name(modality)}.pt")))

    for p in candidates:
        if Path(p).exists():
            return Path(p)

    raise FileNotFoundError(
        f"Missing CNN-BiLSTM Real->Real model for modality '{modality}'. "
        f"Expected pretrained models under: {PRETRAINED_RESULTS_DIR / 'cnnbilstm_saved_models'}"
    )


def save_checkpoint(model, model_path, modality, view, best_epoch, best_val_metrics):
    model_path = Path(model_path)
    model_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "framework": "cnnbilstm",
            "model": CNN_MODEL_NAME,
            "modality": modality,
            "branch_names": list(view["branch_names"]),
            "native_hz": view["native_hz"],
            "channels": view["channels"],
            "activity_ids": ACTIVITY_IDS,
            "num_classes": NUM_CLASSES,
            "n_seq": N_SEQ,
            "branch_specs": {
                k: {
                    "seq_len": BRANCH_SPECS[k]["seq_len"],
                    "in_channels": BRANCH_SPECS[k]["in_channels"],
                    "expected_shape_tail": BRANCH_SPECS[k]["expected_shape_tail"],
                    "native_hz": BRANCH_SPECS[k]["native_hz"],
                    "channel_names": BRANCH_SPECS[k]["channel_names"],
                }
                for k in view["branch_names"]
            },
            "best_epoch": best_epoch,
            "best_val_metrics": best_val_metrics,
            "batch_size": BATCH_SIZE,
            "lr": LR,
            "weight_decay": WEIGHT_DECAY,
            "dropout": DROPOUT,
            "use_amp": USE_AMP,
            "use_tf32": USE_TF32,
            "seed": SEED,
            "train_subjects": TRAIN_SUBJECTS,
            "val_subjects": VAL_SUBJECTS,
            "test_subjects": TEST_SUBJECTS,
        },
        model_path,
    )
    print("Saved CNN-BiLSTM model:", model_path)
    return str(model_path)


def load_saved_cnnbilstm_model(model_file, view):
    model_file = Path(model_file)
    ckpt = torch.load(model_file, map_location=DEVICE)
    ckpt_branches = list(ckpt.get("branch_names", []))
    expected_branches = list(view["branch_names"])
    if ckpt_branches and ckpt_branches != expected_branches:
        raise ValueError(
            f"Saved model branch mismatch. File has {ckpt_branches}, expected {expected_branches}. File: {model_file}"
        )
    model = make_new_model(expected_branches).to(DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    return model, ckpt


def train_cnnbilstm(exp, view, modality):
    branch_names = view["branch_names"]
    train_loader = make_loader(exp["train_X"], exp["train_y"], branch_names, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = make_loader(exp["val_X"], exp["val_y"], branch_names, batch_size=BATCH_SIZE, shuffle=False)

    model = make_new_model(branch_names).to(DEVICE)

    if USE_CLASS_WEIGHTS:
        class_weights = compute_class_weights(exp["train_y"]).to(DEVICE)
        print("\nClass weights:")
        for act, w in zip(ACTIVITY_IDS, class_weights.detach().cpu().numpy()):
            print(f"  Activity {act}: {w:.4f}")
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = make_grad_scaler()

    best_val_macro_f1 = -1.0
    best_state = None
    best_epoch = 0
    best_val_metrics = None
    epochs_without_improvement = 0
    history_rows = []

    for epoch in range(1, EPOCHS_CNNBILSTM + 1):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_seen = 0

        for batch in train_loader:
            x_dict, y_batch = move_batch_to_device(batch)
            optimizer.zero_grad(set_to_none=True)

            with autocast_context():
                logits = model(x_dict)
                loss = criterion(logits, y_batch)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            with torch.no_grad():
                pred = torch.argmax(logits, dim=1)
                correct = (pred == y_batch).sum().item()

            bs = len(y_batch)
            total_loss += float(loss.item()) * bs
            total_correct += correct
            total_seen += bs

        train_loss = total_loss / max(total_seen, 1)
        train_accuracy = total_correct / max(total_seen, 1)

        val_true, val_pred, _ = evaluate_model(model, val_loader)
        val_metrics = compute_classification_metrics(val_true, val_pred)

        best_so_far = max(best_val_macro_f1, val_metrics["macro_f1"])
        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_precision": val_metrics["macro_precision"],
            "val_macro_recall": val_metrics["macro_recall"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"],
            "best_val_macro_f1_so_far": best_so_far,
        }
        history_rows.append(row)

        print(
            f"[cnnbilstm] {exp['name']} | {modality} | "
            f"Epoch {epoch:03d}/{EPOCHS_CNNBILSTM} | "
            f"loss={train_loss:.4f} | "
            f"train_acc={train_accuracy:.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f} | "
            f"val_macro_f1={val_metrics['macro_f1']:.4f} | "
            f"val_bal_acc={val_metrics['balanced_accuracy']:.4f}",
            flush=True,
        )

        if val_metrics["macro_f1"] > best_val_macro_f1:
            best_val_macro_f1 = val_metrics["macro_f1"]
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            best_val_metrics = dict(val_metrics)
            epochs_without_improvement = 0
            print(
                f"Saved new best state in memory | "
                f"epoch={best_epoch} | val_macro_f1={best_val_macro_f1:.4f}",
                flush=True,
            )
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            print(
                f"Early stopping at epoch {epoch}. "
                f"Best epoch = {best_epoch}. "
                f"Best val macro-F1 = {best_val_macro_f1:.4f}",
                flush=True,
            )
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    history_df = pd.DataFrame(history_rows)
    history_path = OUT_DIR / f"cnnbilstm_{sanitize_name(exp['name'])}_{sanitize_name(modality)}_training_history.csv"
    history_df.to_csv(history_path, index=False)
    print("Saved history:", history_path)

    model_path = get_cnnbilstm_model_file(exp["name"], modality)
    saved_model_file = save_checkpoint(
        model=model,
        model_path=model_path,
        modality=modality,
        view=view,
        best_epoch=best_epoch,
        best_val_metrics=best_val_metrics if best_val_metrics is not None else {},
    )

    return model, {
        "epochs": EPOCHS_CNNBILSTM,
        "best_epoch_by_val_macro_f1": best_epoch,
        "best_val_macro_f1": best_val_macro_f1,
        "best_val_accuracy": None if best_val_metrics is None else best_val_metrics.get("accuracy", None),
        "saved_best_model_file": saved_model_file,
        "training_history_file": str(history_path),
    }


def make_cnnbilstm_row(exp, view, modality, metrics, train_meta, used_real_model=False):
    return {
        "framework": "cnnbilstm",
        "model": CNN_MODEL_NAME,
        "experiment": exp["name"],
        "native_modality": modality,
        "native_hz": view["native_hz"],
        "channels": channel_string(view["channels"]),
        "branches": ",".join(view["branch_names"]),
        "description": view.get("description", ""),
        "input_shape_train_native_by_branch_N_T_C": shape_dict_string(exp["train_X"]),
        "input_shape_test_native_by_branch_N_T_C": shape_dict_string(exp["test_X"]),
        "input_shape_train_framework": "dict_of_native_rate_N_T_C_tensors",
        "input_shape_test_framework": "dict_of_native_rate_N_T_C_tensors",
        "epochs": train_meta.get("epochs", 0),
        "best_epoch_by_val_macro_f1": train_meta.get("best_epoch_by_val_macro_f1", None),
        "best_val_macro_f1": train_meta.get("best_val_macro_f1", None),
        "best_val_accuracy": train_meta.get("best_val_accuracy", None),
        "saved_best_model_file": str(train_meta.get("saved_best_model_file", "")),
        "training_history_file": str(train_meta.get("training_history_file", "")),
        "used_saved_real_to_real_model": bool(used_real_model),
        "training_reused_from_experiment": "real_to_real" if used_real_model else "",
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "dropout": DROPOUT,
        "patience": PATIENCE,
        **metrics,
    }


def run_cnnbilstm(views, modalities):
    rows = []
    activity_rows = []

    for modality in modalities:
        if modality not in views:
            print(f"[cnnbilstm] Skipping unknown modality: {modality}")
            continue

        view = views[modality]
        for exp in get_experiments_for_view(view):
            print("\n" + "=" * 80)
            print(f"[cnnbilstm] {exp['name']} | {modality}")
            print("=" * 80)
            print("Branches:", view["branch_names"])
            print("Train shapes:", shape_dict(exp["train_X"]))
            print("Val shapes:  ", shape_dict(exp["val_X"]))
            print("Test shapes: ", shape_dict(exp["test_X"]))
            print("Train label counts:", dict(zip(*np.unique(exp["train_y"], return_counts=True))))
            print("Val label counts:  ", dict(zip(*np.unique(exp["val_y"], return_counts=True))))
            print("Test label counts: ", dict(zip(*np.unique(exp["test_y"], return_counts=True))))

            if exp["name"] == "real_to_real" and not RUN_REAL_TO_REAL_ONLY:
                existing = get_existing_result_row("cnnbilstm", "real_to_real", modality, model=CNN_MODEL_NAME)
                if existing is not None:
                    print("Reusing saved CNN-BiLSTM Real->Real metrics.")
                    rows.append(existing)
                    existing_activity = get_existing_activity_rows("cnnbilstm", "real_to_real", modality, model=CNN_MODEL_NAME)
                    if len(existing_activity) > 0:
                        activity_rows.append(existing_activity)
                    continue

            try:
                set_seed(SEED)

                if exp["name"] == "real_to_synthetic" and USE_REAL_MODELS:
                    model_file = find_cnnbilstm_real_model_file(modality)
                    print("Loading CNN-BiLSTM Real->Real model:", model_file)
                    model, ckpt = load_saved_cnnbilstm_model(model_file, view)
                    best_val_metrics = ckpt.get("best_val_metrics", {}) or {}
                    train_meta = {
                        "epochs": 0,
                        "best_epoch_by_val_macro_f1": ckpt.get("best_epoch", None),
                        "best_val_macro_f1": best_val_metrics.get("macro_f1", None),
                        "best_val_accuracy": best_val_metrics.get("accuracy", None),
                        "saved_best_model_file": str(model_file),
                        "training_history_file": "",
                    }
                    used_real_model = True
                else:
                    model, train_meta = train_cnnbilstm(exp, view, modality)
                    used_real_model = False

                test_loader = make_loader(
                    exp["test_X"], exp["test_y"], view["branch_names"], batch_size=BATCH_SIZE, shuffle=False
                )
                y_true, y_pred, y_prob = evaluate_model(model, test_loader)
                metrics = compute_classification_metrics(y_true, y_pred)

                activity_df = save_predictions_cm_and_activity_report(
                    "cnnbilstm",
                    CNN_MODEL_NAME,
                    exp["name"],
                    modality,
                    view["native_hz"],
                    view["channels"],
                    y_true,
                    y_pred,
                    y_prob=y_prob,
                )
                activity_rows.append(activity_df)

                row = make_cnnbilstm_row(
                    exp, view, modality, metrics, train_meta, used_real_model=used_real_model
                )
                rows.append(row)
                print(json.dumps(row, indent=2))

            except Exception as e:
                warnings.warn(f"[cnnbilstm] Failed {exp['name']} | {modality}: {repr(e)}")
                rows.append({
                    "framework": "cnnbilstm",
                    "model": CNN_MODEL_NAME,
                    "experiment": exp["name"],
                    "native_modality": modality,
                    "error": repr(e),
                })

            finally:
                try:
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                except Exception:
                    pass

    df = pd.DataFrame(rows)
    df.to_csv(result_csv_path("cnnbilstm"), index=False)
    activity_df_all = pd.concat(activity_rows, ignore_index=True) if activity_rows else pd.DataFrame()
    activity_df_all.to_csv(activity_csv_path("cnnbilstm"), index=False)
    print("\n[cnnbilstm] Saved:", result_csv_path("cnnbilstm"))
    print("[cnnbilstm] Saved:", activity_csv_path("cnnbilstm"))
    return df, activity_df_all


# =========================
# RUN AND COVERAGE
# =========================

def expected_experiments():
    if RUN_REAL_TO_REAL_ONLY:
        return ["real_to_real"]
    return ["real_to_real", "real_to_synthetic", "synthetic_to_real", "real_plus_synthetic_to_real"]


def save_coverage_report(final_results_df):
    expected = pd.DataFrame(
        list(itertools.product(["cnnbilstm"], expected_experiments(), MODALITIES)),
        columns=["framework", "experiment", "native_modality"],
    )

    if len(final_results_df) > 0:
        df = final_results_df.copy()
        if "error" not in df.columns:
            df["error"] = ""
        df["has_error"] = df["error"].fillna("").astype(str) != ""
        actual = (
            df.groupby(["framework", "experiment", "native_modality"], dropna=False)
            .agg(row_count=("framework", "size"), error_count=("has_error", "sum"))
            .reset_index()
        )
    else:
        actual = pd.DataFrame(columns=["framework", "experiment", "native_modality", "row_count", "error_count"])

    coverage = expected.merge(actual, on=["framework", "experiment", "native_modality"], how="left")
    coverage["row_count"] = coverage["row_count"].fillna(0).astype(int)
    coverage["error_count"] = coverage["error_count"].fillna(0).astype(int)
    coverage["status"] = np.where(
        coverage["row_count"] == 0,
        "missing",
        np.where(coverage["error_count"] > 0, "error", "ok"),
    )

    path = OUT_DIR / "experiment_coverage_report.csv"
    coverage.to_csv(path, index=False)
    print("Saved coverage report:", path)
    print("Expected rows:", len(expected))
    print("Actual rows:", len(final_results_df))
    print("Rows with errors:", int(coverage["error_count"].sum()))
    return coverage


def main():
    set_seed(SEED)

    print("=" * 80)
    print("Native CNN-BiLSTM evaluation with tsai/aeon-style outputs")
    print("=" * 80)
    print("SELECTED_MODEL_FAMILY:", SELECTED_MODEL_FAMILY)
    print("SELECTED_SYNTHETIC_METHOD:", SELECTED_SYNTHETIC_METHOD)
    print("RUN_REAL_TO_REAL_ONLY:", RUN_REAL_TO_REAL_ONLY)
    print("USE_REAL_MODELS:", USE_REAL_MODELS)
    print("MODALITIES:", MODALITIES)
    print("OUT_DIR:", OUT_DIR)
    print("REAL_DIR:", REAL_DIR)
    print("SYN_DIR:", SYN_DIR)
    print("PRETRAINED_RESULTS_DIR:", PRETRAINED_RESULTS_DIR)
    print("Device:", DEVICE)
    if DEVICE == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))
    print("BATCH_SIZE:", BATCH_SIZE)
    print("NUM_WORKERS:", NUM_WORKERS)
    print("EPOCHS_CNNBILSTM:", EPOCHS_CNNBILSTM)
    print("USE_AMP:", USE_AMP)
    print("USE_TF32:", USE_TF32)
    print("SEED:", SEED)

    load_synthetic = not RUN_REAL_TO_REAL_ONLY
    real_X, real_y, real_subjects, syn_X, syn_y, syn_subjects = load_native_data(load_synthetic=load_synthetic)

    views = make_native_dataset_views(real_X, real_y, real_subjects, syn_X, syn_y, syn_subjects)
    shape_report_df = save_shape_report(views)
    safe_display(shape_report_df)

    final_results, final_activity = run_cnnbilstm(views, MODALITIES)

    final_results_path = OUT_DIR / "all_framework_native_results.csv"
    final_activity_path = OUT_DIR / "all_framework_native_per_activity_results.csv"
    final_results.to_csv(final_results_path, index=False)
    final_activity.to_csv(final_activity_path, index=False)
    coverage = save_coverage_report(final_results)

    print("\n" + "=" * 80)
    print("DONE")
    print("=" * 80)
    print("Saved aggregate results:", result_csv_path("cnnbilstm"))
    print("Saved aggregate per-activity results:", activity_csv_path("cnnbilstm"))
    print("Saved combined-style results:", final_results_path)
    print("Saved combined-style per-activity results:", final_activity_path)
    print("Saved models in:", OUT_DIR / "cnnbilstm_saved_models")

    print("\nAggregate results:")
    safe_display(final_results)
    print("\nCoverage:")
    safe_display(coverage)




## Run

First unzip your friend's pretrained folder so the final structure is:

```text
/home/iailab42/khans1/projects/ir/models/downstream/pretrained/cnnbilstm_native_results/
  cnnbilstm_saved_models/
  cnnbilstm_native_downstream_results.csv
  cnnbilstm_native_per_activity_results.csv
```

Then set:

```python
SELECTED_MODEL_FAMILY = "kovae"
SELECTED_SYNTHETIC_METHOD = "posterior_bank_v2"
```

or:

```python
SELECTED_MODEL_FAMILY = "kovae"
SELECTED_SYNTHETIC_METHOD = "rollout_v1"
```

or:

```python
SELECTED_MODEL_FAMILY = "timevae"
SELECTED_SYNTHETIC_METHOD = "prior_v1"
```


In [4]:
# Run exactly one selected synthetic method.
# Change SELECTED_MODEL_FAMILY and SELECTED_SYNTHETIC_METHOD in the first code cell before running this.

main()


Native CNN-BiLSTM evaluation with tsai/aeon-style outputs
SELECTED_MODEL_FAMILY: timevae
SELECTED_SYNTHETIC_METHOD: prior_v1
RUN_REAL_TO_REAL_ONLY: False
USE_REAL_MODELS: True
MODALITIES: ['acc', 'bvp', 'eda', 'temp', 'fused']
OUT_DIR: /home/iailab42/khans1/projects/ir/results/downstream_cnnbilstm/timevae/prior_v1
REAL_DIR: /home/iailab42/khans1/projects/ir/data/processed/native_rates
SYN_DIR: /home/iailab42/khans1/projects/ir/data/synthetic_subjects/timevae/prior_v1
PRETRAINED_RESULTS_DIR: /home/iailab42/khans1/projects/ir/models/downstream/pretrained/cnnbilstm_native_results
Device: cuda
GPU: NVIDIA RTX A6000
BATCH_SIZE: 384
NUM_WORKERS: 8
EPOCHS_CNNBILSTM: 100
USE_AMP: True
USE_TF32: True
SEED: 42

Loaded real data:
  y: (46925,)
  subjects: (46925,)
  acc: (46925, 256, 3)
  bvp: (46925, 512, 1)
  slow: (46925, 32, 2)

Loaded synthetic data:
  y: (30000,)
  subjects: (30000,)
  acc: (30000, 256, 3)
  bvp: (30000, 512, 1)
  slow: (30000, 32, 2)

Fixed split OK:
  Train: ['S1', 'S2', 

,native_view,native_hz,channels,branches,description,native_real_train_shape_by_branch_N_T_C,native_real_val_shape_by_branch_N_T_C,native_real_test_shape_by_branch_N_T_C,framework_input_style,native_syn_all_shape_by_branch_N_T_C,native_syn_test3_shape_by_branch_N_T_C,syn_test_subjects
0,acc,32,"ACC_x,ACC_y,ACC_z",acc,ACC only,"{""acc"": [30762, 256, 3]}","{""acc"": [6100, 256, 3]}","{""acc"": [10063, 256, 3]}",dict_of_native_rate_N_T_C_tensors,"{""acc"": [30000, 256, 3]}","{""acc"": [30000, 256, 3]}","synthetic_subject_01,synthetic_subject_02,synt..."
1,bvp,64,BVP,bvp,BVP only,"{""bvp"": [30762, 512, 1]}","{""bvp"": [6100, 512, 1]}","{""bvp"": [10063, 512, 1]}",dict_of_native_rate_N_T_C_tensors,"{""bvp"": [30000, 512, 1]}","{""bvp"": [30000, 512, 1]}","synthetic_subject_01,synthetic_subject_02,synt..."
2,eda,4,EDA,eda,EDA only,"{""eda"": [30762, 32, 1]}","{""eda"": [6100, 32, 1]}","{""eda"": [10063, 32, 1]}",dict_of_native_rate_N_T_C_tensors,"{""eda"": [30000, 32, 1]}","{""eda"": [30000, 32, 1]}","synthetic_subject_01,synthetic_subject_02,synt..."
3,temp,4,TEMP,temp,TEMP only,"{""temp"": [30762, 32, 1]}","{""temp"": [6100, 32, 1]}","{""temp"": [10063, 32, 1]}",dict_of_native_rate_N_T_C_tensors,"{""temp"": [30000, 32, 1]}","{""temp"": [30000, 32, 1]}","synthetic_subject_01,synthetic_subject_02,synt..."
4,fused,32/64/4/4,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP","acc,bvp,eda,temp",ACC+BVP+EDA+TEMP fused native-rate multibranch,"{""acc"": [30762, 256, 3], ""bvp"": [30762, 512, 1...","{""acc"": [6100, 256, 3], ""bvp"": [6100, 512, 1],...","{""acc"": [10063, 256, 3], ""bvp"": [10063, 512, 1...",dict_of_native_rate_N_T_C_tensors,"{""acc"": [30000, 256, 3], ""bvp"": [30000, 512, 1...","{""acc"": [30000, 256, 3], ""bvp"": [30000, 512, 1...","synthetic_subject_01,synthetic_subject_02,synt..."



[cnnbilstm] real_to_real | acc
Branches: ['acc']
Train shapes: {'acc': [30762, 256, 3]}
Val shapes:   {'acc': [6100, 256, 3]}
Test shapes:  {'acc': [10063, 256, 3]}
Train label counts: {np.int64(1): np.int64(3032), np.int64(2): np.int64(2139), np.int64(3): np.int64(1500), np.int64(4): np.int64(2296), np.int64(5): np.int64(4580), np.int64(6): np.int64(8792), np.int64(7): np.int64(2903), np.int64(8): np.int64(5520)}
Val label counts:   {np.int64(1): np.int64(605), np.int64(2): np.int64(432), np.int64(3): np.int64(335), np.int64(4): np.int64(449), np.int64(5): np.int64(866), np.int64(6): np.int64(1581), np.int64(7): np.int64(632), np.int64(8): np.int64(1200)}
Test label counts:  {np.int64(1): np.int64(901), np.int64(2): np.int64(635), np.int64(3): np.int64(444), np.int64(4): np.int64(697), np.int64(5): np.int64(1363), np.int64(6): np.int64(3147), np.int64(7): np.int64(1128), np.int64(8): np.int64(1748)}
Reusing saved CNN-BiLSTM Real->Real metrics.

[cnnbilstm] real_to_synthetic | acc
Bra

,framework,model,experiment,native_modality,native_hz,channels,branches,description,input_shape_train_native_by_branch_N_T_C,input_shape_test_native_by_branch_N_T_C,...,dropout,patience,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1,balanced_accuracy
0,cnnbilstm,NativeCNNBiLSTM,real_to_real,acc,32,"ACC_x,ACC_y,ACC_z",acc,ACC only,"{""acc"": [30762, 256, 3]}","{""acc"": [10063, 256, 3]}",...,0.5,25,0.540992,0.616012,0.583939,0.586813,0.573569,0.540992,0.543841,0.583939
1,cnnbilstm,NativeCNNBiLSTM,real_to_synthetic,acc,32,"ACC_x,ACC_y,ACC_z",acc,ACC only,"{""acc"": [30762, 256, 3]}","{""acc"": [30000, 256, 3]}",...,0.5,25,0.320067,0.374442,0.160538,0.121590,0.350856,0.320067,0.200049,0.160538
2,cnnbilstm,NativeCNNBiLSTM,synthetic_to_real,acc,32,"ACC_x,ACC_y,ACC_z",acc,ACC only,"{""acc"": [30000, 256, 3]}","{""acc"": [10063, 256, 3]}",...,0.5,25,0.317003,0.320792,0.335363,0.289459,0.375982,0.317003,0.303158,0.335363
3,cnnbilstm,NativeCNNBiLSTM,real_plus_synthetic_to_real,acc,32,"ACC_x,ACC_y,ACC_z",acc,ACC only,"{""acc"": [60762, 256, 3]}","{""acc"": [10063, 256, 3]}",...,0.5,25,0.578555,0.638220,0.650933,0.638185,0.598007,0.578555,0.584355,0.650933
4,cnnbilstm,NativeCNNBiLSTM,real_to_real,bvp,64,BVP,bvp,BVP only,"{""bvp"": [30762, 512, 1]}","{""bvp"": [10063, 512, 1]}",...,0.5,25,0.403657,0.406051,0.399845,0.392110,0.396835,0.403657,0.392425,0.399845
5,cnnbilstm,NativeCNNBiLSTM,real_to_synthetic,bvp,64,BVP,bvp,BVP only,"{""bvp"": [30762, 512, 1]}","{""bvp"": [30000, 512, 1]}",...,0.5,25,0.206433,0.172752,0.165855,0.146564,0.210084,0.206433,0.188165,0.165855
6,cnnbilstm,NativeCNNBiLSTM,synthetic_to_real,bvp,64,BVP,bvp,BVP only,"{""bvp"": [30000, 512, 1]}","{""bvp"": [10063, 512, 1]}",...,0.5,25,0.156216,0.162767,0.157323,0.136916,0.216083,0.156216,0.160206,0.157323
7,cnnbilstm,NativeCNNBiLSTM,real_plus_synthetic_to_real,bvp,64,BVP,bvp,BVP only,"{""bvp"": [60762, 512, 1]}","{""bvp"": [10063, 512, 1]}",...,0.5,25,0.385571,0.400318,0.436571,0.389707,0.405543,0.385571,0.373694,0.436571
8,cnnbilstm,NativeCNNBiLSTM,real_to_real,eda,4,EDA,eda,EDA only,"{""eda"": [30762, 32, 1]}","{""eda"": [10063, 32, 1]}",...,0.5,25,0.216536,0.179476,0.288770,0.176426,0.167432,0.216536,0.148211,0.288770
9,cnnbilstm,NativeCNNBiLSTM,real_to_synthetic,eda,4,EDA,eda,EDA only,"{""eda"": [30762, 32, 1]}","{""eda"": [30000, 32, 1]}",...,0.5,25,0.146867,0.186103,0.168831,0.123455,0.186942,0.146867,0.125620,0.168831



Coverage:


,framework,experiment,native_modality,row_count,error_count,status
0,cnnbilstm,real_to_real,acc,1,0,ok
1,cnnbilstm,real_to_real,bvp,1,0,ok
2,cnnbilstm,real_to_real,eda,1,0,ok
3,cnnbilstm,real_to_real,temp,1,0,ok
4,cnnbilstm,real_to_real,fused,1,0,ok
5,cnnbilstm,real_to_synthetic,acc,1,0,ok
6,cnnbilstm,real_to_synthetic,bvp,1,0,ok
7,cnnbilstm,real_to_synthetic,eda,1,0,ok
8,cnnbilstm,real_to_synthetic,temp,1,0,ok
9,cnnbilstm,real_to_synthetic,fused,1,0,ok
